# Data Storytelling with Amazon Sales

Turn a small sales dataset into an evidence-based business story. We will move from a question to analysis, visualization, insight, and recommendation.

## Learning objectives

- Validate and prepare a dataset.
- Select charts that match the analytical question.
- Calculate insights instead of hard-coding conclusions.
- Communicate a concise recommendation with appropriate limitations.

In [ ]:
# Step 1: Import libraries
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

In [ ]:
# Step 2: Load the CSV included with this repository
data_path = Path('amazon_sales_dataset.csv')
if not data_path.exists():
    raise FileNotFoundError('Place amazon_sales_dataset.csv beside this notebook.')

df = pd.read_csv(data_path)
df.head()

In [ ]:
# Step 3: Validate and prepare the data
required = {'Date', 'Category', 'Sales', 'Profit'}
missing_columns = required.difference(df.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce')
df['Profit'] = pd.to_numeric(df['Profit'], errors='coerce')
df = df.dropna(subset=['Date', 'Category', 'Sales', 'Profit']).sort_values('Date')
print(f'Rows ready for analysis: {len(df)}')
print(df.isna().sum())

In [ ]:
# Step 4: Calculate key performance indicators
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
profit_margin = (total_profit / total_sales * 100) if total_sales else 0

print(f'Total sales: {total_sales:,.2f}')
print(f'Total profit: {total_profit:,.2f}')
print(f'Profit margin: {profit_margin:.1f}%')

In [ ]:
# Step 5: Compare categories
category_summary = (df.groupby('Category', as_index=False)
                    .agg(Sales=('Sales', 'sum'), Profit=('Profit', 'sum'))
                    .sort_values('Sales', ascending=False))
category_summary

In [ ]:
# Step 6: Bar chart — best for category comparison
plt.figure(figsize=(9, 5))
sns.barplot(data=category_summary, x='Category', y='Sales', hue='Category', legend=False, palette='Blues_d')
plt.title('Total Sales by Category')
plt.xlabel('Category')
plt.ylabel('Sales')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
# Step 7: Line chart — best for a time trend
daily_sales = df.groupby('Date', as_index=False)['Sales'].sum()
plt.figure(figsize=(10, 5))
sns.lineplot(data=daily_sales, x='Date', y='Sales', marker='o')
plt.title('Sales Trend Over Time')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Step 8: Scatter plot — investigate the relationship between sales and profit
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='Sales', y='Profit', hue='Category', s=90)
plt.title('Sales vs. Profit')
plt.tight_layout()
plt.show()

correlation = df['Sales'].corr(df['Profit'])
print(f'Sales–profit correlation: {correlation:.2f}')

In [ ]:
# Step 9: Build an evidence-based story from calculated results
top_sales = category_summary.iloc[0]
top_profit = category_summary.sort_values('Profit', ascending=False).iloc[0]
relationship = 'positive' if correlation > 0.3 else 'weak or mixed'

print('DATA STORY')
print(f"1. {top_sales['Category']} leads sales with {top_sales['Sales']:,.2f}.")
print(f"2. {top_profit['Category']} produces the highest profit at {top_profit['Profit']:,.2f}.")
print(f'3. The relationship between sales and profit is {relationship} (r={correlation:.2f}).')
print('Recommendation: protect the strongest category, then investigate margin and trend before increasing marketing spend.')
print('Limitation: this small teaching dataset does not prove causation or guarantee future performance.')

## Student practice

1. Add a profit-by-category chart.
2. Identify one surprising result.
3. Rewrite the recommendation for a non-technical manager.
4. State one limitation of the dataset or analysis.